**Bronze**

In [0]:
from pyspark.sql import functions as F

In [0]:
raw_path = "/Volumes/workspace/default/spotify/raw/spotify_songs.csv"
bronze_path = "/Volumes/workspace/default/spotify/delta/"

df_raw = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv(raw_path)
          )

print(f"Loaded {df_raw.count()} rows and {len(df_raw.columns)} columns")
display(df_raw.limit(5))

Loaded 32833 rows and 23 columns


track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,playlist_subgenre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxury Remix,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury Remix],2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.748,0.916,6,-2.634,1,0.0583,0.102,0.0,0.0653,0.518,122.036,194754.0
0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.726,0.815,11,-4.969,1,0.0373,0.0724,0.00421,0.357,0.693,99.972,162600.0
1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.675,0.931,1,-3.432,0,0.0742,0.0794,2.33E-5,0.11,0.613,124.008,176616.0
75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.718,0.93,7,-3.778,1,0.102,0.0287,9.43E-6,0.204,0.277,121.956,169093.0
1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.65,0.833,1,-4.672,1,0.0359,0.0803,0.0,0.0833,0.725,123.976,189052.0


In [0]:
df_raw.printSchema()

root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- track_artist: string (nullable = true)
 |-- track_popularity: integer (nullable = true)
 |-- track_album_id: string (nullable = true)
 |-- track_album_name: string (nullable = true)
 |-- track_album_release_date: string (nullable = true)
 |-- playlist_name: string (nullable = true)
 |-- playlist_id: string (nullable = true)
 |-- playlist_genre: string (nullable = true)
 |-- playlist_subgenre: string (nullable = true)
 |-- danceability: string (nullable = true)
 |-- energy: string (nullable = true)
 |-- key: string (nullable = true)
 |-- loudness: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- duration_ms: double (nullable = true)


In [0]:
# remove rows with half missing values
print(f"Number of rows before cleaning: {df_raw.count()}")

df_clean = df_raw.dropna(
    thresh=int(len(df_raw.columns) / 2)
)

print(f"Number of rows after removing missing values: {df_clean.count()}")

df_clean = df_clean.dropDuplicates(
    subset=["track_id"]
)

print(f"Number of rows after dropping duplicates: {df_clean.count()}")

Number of rows before cleaning: 32833
Number of rows after removing missing values: 32833
Number of rows after dropping duplicates: 28356


In [0]:
(df_clean.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", True)
 .saveAsTable("workspace.default.bronze_spotify")
 )

In [0]:
spark.sql("""
          SELECT COUNT(*) AS total_rows
          FROM bronze_spotify
          """).show()

+----------+
|total_rows|
+----------+
|     28356|
+----------+



In [0]:
spark.sql("""
          SELECT *
          FROM bronze_spotify
          LIMIT 5
          """).display()

track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,playlist_subgenre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
13qqdlSeF8FcxsRyapDMZ0,Changes,Faul & Wad Ad,39,2TQZYFed2I2ajTSzwZV7ag,Changes,2013-11-15,Dance Pop Tunes,4SdfG4cPG3skmTiQLozZGh,pop,dance pop,0.822,0.704,3,-8.242,0,0.0374,0.00864,0.627,0.0633,0.255,125.999,345173.0
5CtI0qwDJkDQGwXD1H1cLb,Despacito - Remix,Luis Fonsi,24,3smvpv7CdrhVcGYaNDLOqn,Despacito Feat. Justin Bieber (Remix),2017-04-17,Electropop Hits 2017-2020,7kyvBmlc1uSqsTL0EuNLrx,pop,electropop,0.694,0.815,2,-4.328,1,0.12,0.229,0.0,0.0924,0.813,88.931,228827.0
2GRL794l2R1SFJc3Z0pIlr,Away,The Bolshoi,0,1IZ1slyJWQT1BPfrX1u5Bb,Friends,1986,"Maxi Pop GOLD (New Wave, Electropop, Synth Pop...)",2nRWtTI9a2LWjJ9Wy3JZs5,pop,electropop,0.6,0.752,0,-8.984,1,0.0358,0.165,7.12E-6,0.157,0.541,121.079,295360.0
3GeeyHU1Q6xjyeXjRAjBoo,Reality (feat. Janieck Devy) - Radio Edit,Lost Frequencies,57,1o18e1NP68Oe5UWA0JuxI2,Reality (feat. Janieck Devy),2015-05-24,ElectroPop,0cuHKz65ZPqBX1brG8djlg,pop,electropop,0.731,0.639,9,-6.597,1,0.0359,0.0206,1.29E-5,0.0789,0.527,122.08,158474.0
5FTCKvxzqy72ceS4Ujux4N,What's My Name?,Rihanna,68,7vN82vd1Vq44fjlhjfvHJp,Loud,2010-11-16,10er Playlist,1kEczIkZH8IgaWT2BiApxZ,pop,electropop,0.69,0.786,2,-2.959,1,0.0692,0.229,0.0,0.0797,0.583,100.049,263173.0


**Silver**

In [0]:
from pyspark.sql.functions import col, when, current_timestamp, regexp_replace

In [0]:
silver_path = "/Volumes/workspace/default/spotify/delta/"

In [0]:
df_silver = df_clean

In [0]:
# cast columns to proper types
df_silver = df_silver \
    .withColumn("danceability", F.expr("try_cast(danceability AS DOUBLE)")) \
    .withColumn("energy", F.expr("try_cast(energy AS DOUBLE)")) \
    .withColumn("loudness", F.expr("try_cast(loudness AS DOUBLE)")) \
    .withColumn("key", F.expr("try_cast(key AS INT)")) \
    .withColumn("mode", F.expr("try_cast(mode AS INT)")) \
    .withColumn("duration_s", F.expr("try_cast(duration_ms AS DOUBLE) / 1000")) \
    .withColumn("album_release_date", F.expr("try_cast(track_album_release_date AS DATE)"))

In [0]:
# create duration flag
df_silver = df_silver.withColumn("duration_flag", when(col("duration_s") >= 180, 1).otherwise(0))

In [0]:
audio_cols = ["danceability","energy","loudness","speechiness","acousticness",
              "instrumentalness","liveness","valence","tempo","duration_s"]

# replace null numeric audio features with 0
for c in audio_cols:
    df_silver = df_silver.fillna({c:0})

In [0]:
(df_silver.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable("workspace.default.silver_spotify")
)

In [0]:
spark.sql("""
          SELECT COUNT(*) AS total_rows
          FROM silver_spotify
          """).show()

+----------+
|total_rows|
+----------+
|     28356|
+----------+



In [0]:
spark.sql("""
          SELECT *
          FROM silver_spotify
          LIMIT 5
          """).display()

track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,playlist_subgenre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,duration_s,album_release_date,duration_flag
13qqdlSeF8FcxsRyapDMZ0,Changes,Faul & Wad Ad,39,2TQZYFed2I2ajTSzwZV7ag,Changes,2013-11-15,Dance Pop Tunes,4SdfG4cPG3skmTiQLozZGh,pop,dance pop,0.822,0.704,3,-8.242,0,0.0374,0.00864,0.627,0.0633,0.255,125.999,345173.0,345.173,2013-11-15,1
5CtI0qwDJkDQGwXD1H1cLb,Despacito - Remix,Luis Fonsi,24,3smvpv7CdrhVcGYaNDLOqn,Despacito Feat. Justin Bieber (Remix),2017-04-17,Electropop Hits 2017-2020,7kyvBmlc1uSqsTL0EuNLrx,pop,electropop,0.694,0.815,2,-4.328,1,0.12,0.229,0.0,0.0924,0.813,88.931,228827.0,228.827,2017-04-17,1
2GRL794l2R1SFJc3Z0pIlr,Away,The Bolshoi,0,1IZ1slyJWQT1BPfrX1u5Bb,Friends,1986,"Maxi Pop GOLD (New Wave, Electropop, Synth Pop...)",2nRWtTI9a2LWjJ9Wy3JZs5,pop,electropop,0.6,0.752,0,-8.984,1,0.0358,0.165,7.12E-6,0.157,0.541,121.079,295360.0,295.36,1986-01-01,1
3GeeyHU1Q6xjyeXjRAjBoo,Reality (feat. Janieck Devy) - Radio Edit,Lost Frequencies,57,1o18e1NP68Oe5UWA0JuxI2,Reality (feat. Janieck Devy),2015-05-24,ElectroPop,0cuHKz65ZPqBX1brG8djlg,pop,electropop,0.731,0.639,9,-6.597,1,0.0359,0.0206,1.29E-5,0.0789,0.527,122.08,158474.0,158.474,2015-05-24,0
5FTCKvxzqy72ceS4Ujux4N,What's My Name?,Rihanna,68,7vN82vd1Vq44fjlhjfvHJp,Loud,2010-11-16,10er Playlist,1kEczIkZH8IgaWT2BiApxZ,pop,electropop,0.69,0.786,2,-2.959,1,0.0692,0.229,0.0,0.0797,0.583,100.049,263173.0,263.173,2010-11-16,1


**Gold**

In [0]:
from pyspark.sql.functions import col, round, count, avg, year, when

In [0]:
gold_path = "/Volumes/workspace/default/spotify/delta/"

In [0]:
df_gold = df_silver

In [0]:
# extract release year
df_gold = df_gold.withColumn("release_year", year(col("album_release_date")))

In [0]:
df_gold = df_gold.withColumn(
    "popularity_band",
    when(col("track_popularity") >= 80, "High")
    .when(col("track_popularity") >= 50, "Medium")
    .otherwise("Low")
)

In [0]:
# artist-level aggregation
df_artist_gold = df_gold.groupBy("track_artist") \
    .agg(
        count("*").alias("total_tracks"),
        round(avg("danceability"), 2).alias("avg_danceability"),
        round(avg("energy"), 2).alias("avg_energy"),
        round(avg("valence"), 2).alias("avg_valence"),
        round(avg("track_popularity"), 2).alias("avg_popularity")
    )

# save as gold delta table
(df_artist_gold.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable("workspace.default.artist_gold_spotify")
)


In [0]:
spark.sql("""
          SELECT *
          FROM artist_gold_spotify
          ORDER BY total_tracks DESC
          LIMIT 5
          """).display()

track_artist,total_tracks,avg_danceability,avg_energy,avg_valence,avg_popularity
Queen,130,0.48,0.62,0.46,42.4
Martin Garrix,87,0.59,0.8,0.25,41.4
Don Omar,84,0.73,0.8,0.72,39.06
David Guetta,81,0.62,0.8,0.43,49.37
Dimitri Vegas & Like Mike,68,0.58,0.9,0.32,36.22


In [0]:
# album-level aggregation
df_album_gold = df_gold.groupBy("track_album_id", "track_album_name", "track_artist") \
    .agg(
        count("*").alias("track_count"),
        round(avg("duration_s"),2).alias("avg_duration_s"),
        round(avg("energy"),2).alias("avg_energy"),
        round(avg("danceability"),2).alias("avg_danceability")
    )

# save as gold delta table
(df_album_gold.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable("workspace.default.album_gold_spotify")
)

In [0]:
spark.sql("""
          SELECT *
          FROM album_gold_spotify
          ORDER BY track_count DESC
          LIMIT 5
          """).display()

track_album_id,track_album_name,track_artist,track_count,avg_duration_s,avg_energy,avg_danceability
5L1xcowSxwzFUSJzvyMp48,Ultimate Freestyle Mega Mix,Ballin Entertainment,42,164.35,0.72,0.69
5fstCqs5NpIlF42VhPNv23,Rock & Rios (Remastered),Miguel Rios,29,201.36,0.84,0.41
5XCOTqG63V60nS82PmqMBe,Asian Dreamer,CASIOPEA,20,268.85,0.59,0.63
246E5OvV4QXhPrGOSj7vdb,Trip Stories,Rob Stepwart,20,136.45,0.77,0.72
6TBwXfQCeLoVIOW53dNLqz,The Complete Collection,Lynyrd Skynyrd,16,265.46,0.64,0.47


In [0]:
# playlist-level aggregation
df_playlist_gold = df_gold.groupBy("playlist_id", "playlist_name", "playlist_genre", "playlist_subgenre") \
    .agg(
        count("*").alias("track_count"),
        round(avg("track_popularity"),2).alias("avg_popularity"),
        round(avg("danceability"),2).alias("avg_danceability"),
        round(avg("energy"),2).alias("avg_energy")
    )

# save as gold delta table
(df_playlist_gold.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable("workspace.default.playlist_gold_spotify")
)

In [0]:
spark.sql("""
          SELECT *
          FROM playlist_gold_spotify
          ORDER BY avg_popularity DESC
          LIMIT 5
          """).display()

playlist_id,playlist_name,playlist_genre,playlist_subgenre,track_count,avg_popularity,avg_danceability,avg_energy
Feeling Accomplished,2016-12-02,37i9dQZF1DWTDafB3skWPN,r&b,1,81.0,0.0,0.74
37i9dQZF1DX0XUsuxWHRQd,RapCaviar,rap,hip hop,46,79.2,0.77,0.63
46Cl6dmeiylK6TRGXr7hHe,Pop - Pop UK - 2019 - Canadian Pop - 2019 - Pop,pop,post-teen pop,65,78.29,0.7,0.65
6o6MNYZqHSkMAKcCHPNu7K,Intro to Post-Teen Pop,pop,post-teen pop,16,77.44,0.67,0.61
2ji5tRQVfnhaX1w9FhmSzk,Todo Éxitos,pop,dance pop,60,76.83,0.69,0.69
